# Dedup and Survivorship

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/02_core_patterns/dedup_survivorship/playbook.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/02_core_patterns/dedup_survivorship/playbook.ipynb)

## Business Scenario

CRMs and event streams often produce multiple updates for the same customer. Without survivorship rules, metrics inflate and teams act on stale attributes.

## Value Proposition

- Enforce one canonical record per entity
- Keep the most recent or highest-quality version
- Reduce downstream noise in analytics

---

## Goals

1. Sort updates by timestamp
2. Keep the best record per customer
3. Return a clean, deduplicated dataset


In [1]:
from lakelogic import DataProcessor


In [2]:
from lakelogic import DataProcessor
import csv

# Load updates using the standard library CSV reader
with open("data/customer_updates.csv", newline="") as handle:
    data = list(csv.DictReader(handle))

processor = DataProcessor(contract="contract.yaml")
result = processor.run(data)

raw_df = result.raw
good_df = result.good
bad_df = result.bad

processor.last_report


2026-02-15 01:55:35.984 | INFO     | lakelogic.core.processor:run:278 - Starting LakeLogic run [Auto-Engine: polars, Contract: Dedup & Survivorship]
2026-02-15 01:55:36.044 | INFO     | lakelogic.core.processor:run:319 - Run complete. Source: 5, Total (post-transform): 3, Good: 3, Quarantined: 0, Pre-Transform Dropped: 2, Ratio: 0.00%
2026-02-15 01:55:36.047 | WARNING  | lakelogic.core.processor:run:345 - Schema drift detected for 'Dedup & Survivorship': missing=['is_active'], unknown=[]


{'run_id': 'bb355de47fa44a05a521097e3d7aba87',
 'pipeline_run_id': None,
 'engine': 'polars',
 'contract': 'Dedup & Survivorship',
 'stage': 'default',
 'dataset': 'silver_crm_customer_updates',
 'domain': None,
 'system': None,
 'data_layer': None,
 'source_path': None,
 'source_files': [],
 'max_source_mtime': None,
 'timestamp': '2026-02-15T01:55:37+00:00',
 'counts': {'source': 5,
  'total': 3,
  'good': 3,
  'quarantined': 0,
  'quarantine_ratio': 0.0,
  'pre_transform_dropped': 2},
 'dataset_rules': [],
 'slos': {},
 'row_rule_failures': [],
 'schema_drift': {'missing_fields': ['is_active'],
  'unknown_fields': [],
  'policy': 'drop',
  'evolution': '',
  'allow_schema_drift': True}}

In [3]:
print("GOOD DATA")
print(good_df)

print("BAD DATA")
print(bad_df)


GOOD DATA
shape: (3, 9)
┌───────────┬───────────┬──────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ customer_ ┆ email     ┆ status   ┆ updated_a ┆ … ┆ is_active ┆ _lakelogi ┆ _lakelogi ┆ _lakelogi │
│ id        ┆ ---       ┆ ---      ┆ t         ┆   ┆ ---       ┆ c_source  ┆ c_process ┆ c_run_id  │
│ ---       ┆ str       ┆ str      ┆ ---       ┆   ┆ bool      ┆ ---       ┆ ed_at     ┆ ---       │
│ i64       ┆           ┆          ┆ date      ┆   ┆           ┆ null      ┆ ---       ┆ str       │
│           ┆           ┆          ┆           ┆   ┆           ┆           ┆ str       ┆           │
╞═══════════╪═══════════╪══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1         ┆ alice@exa ┆ inactive ┆ 2024-02-0 ┆ … ┆ false     ┆ null      ┆ 2026-02-1 ┆ bb355de47 │
│           ┆ mple.com  ┆          ┆ 1         ┆   ┆           ┆           ┆ 5T01:55:3 ┆ fa44a05a5 │
│           ┆           ┆          ┆           ┆   ┆           ┆   